# ⚡ Optimized Historical Weather Data Extraction
**Cluster:** Standard_D8s_v3 — 32 GB RAM, 8 Cores (driver)

| Optimization | Impact |
|---|---|
| Spatial bbox subset before `.load()` | ~100x faster load (seconds vs hours) |
| Vectorized interpolation for all cities at once | ~300x fewer calls |
| 6 parallel workers on 8-core driver | 6x throughput |
| Smart `.idx` handling (no corrupt, no needless delete) | saves ~13 min per re-run |
| Checkpoint file | safe resume on crash |
| Single batched DB push per blob | vs 300 pushes |
| `download_as_bytes()` instead of `download_to_filename()` | avoids Databricks PermissionError |

In [ ]:
# ── Cell 1: Install dependencies (run once, then comment out) ─────────────────
!pip install -q xarray cfgrib eccodes
%restart_python

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import sys, os, tarfile, logging, time, traceback, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from warnings import filterwarnings

import cfgrib
import xarray as xr
import numpy as np
import pandas as pd
from google.cloud import storage

filterwarnings('ignore')

sys.path.append('/Workspace/Power Pricing/PowerDE_Data/')
from functions_UDF1 import fetch_table_in_pandas, push_data

# ── Logging: console + thread-safe file ──────────────────────────────────────
# Background threads cannot write to Jupyter's ZMQ stdout socket safely.
# All threads write to the file; the console handler is for the main thread.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)
_fh = logging.FileHandler('weather_extraction.log', mode='a')
_fh.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s', '%H:%M:%S'))
logging.getLogger().addHandler(_fh)

# Silence noisy cfgrib index warnings
logging.getLogger('cfgrib.messages').setLevel(logging.ERROR)

log.info('Imports complete. Logging to console + weather_extraction.log')

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
BUCKET_NAME       = 'gcp10-sb-shortterm-forecasting'
FOLDER_PATH       = 'nwp/ncmrwf-0p125-re/'
SA_KEY_FILE       = 'agel-svc-shorttermf-qa-storage_bucket_view.json'
DOWNLOAD_DIR      = Path('data_downloads')
CHECKPOINT_FILE   = Path('processed_blobs.txt')
YEAR_FILTER       = '2023'          # set to '' to process all years
TARGET_DS_INDICES = {1, 2, 3, 6}   # dataset indices inside each GRIB to extract
MAX_WORKERS       = 6              # driver has 8 cores; 6 workers + 2 free for OS/main
BBOX_BUFFER       = 1.0            # degrees of lat/lon padding around outermost cities

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

VARIABLES_MAPPING = {
    'v'          : 'northward_wind_50mt',
    'u'          : 'eastward_wind_50mt',
    'u10'        : 'eastward_wind_10mt',
    'v10'        : 'northward_wind_10mt',
    't2m'        : 'air_temperature_2mt',
    'sh2'        : 'specific_humidity_2mt',
    'gh'         : 'gropotential_height',
    'sp'         : 'surface_pressure',
    'prmsl'      : 'pressure_reduce_to_mean_sea_level',
    'avg_sdswrf' : 'solar_radiation',
    'unknown'    : 'rainfall',
}

log.info('Config ready')

In [ ]:
# ── Cell 4: Load cities + compute bounding box ────────────────────────────────
latlong_df = fetch_table_in_pandas('dim_latlong')
latlong_df = latlong_df.reset_index(drop=True)   # ensure clean 0-based index

city_lats = xr.DataArray(latlong_df['lat'].astype(float).values, dims='city')
city_lons = xr.DataArray(latlong_df['long'].astype(float).values, dims='city')

# City metadata lookup (index = position in latlong_df)
city_meta = latlong_df[['City', 'State']].copy()
city_meta.index.name = 'city'
city_meta = city_meta.reset_index()   # columns: city, City, State

# India bounding box — auto-derived from your cities + buffer
LAT_MIN = float(city_lats.min()) - BBOX_BUFFER
LAT_MAX = float(city_lats.max()) + BBOX_BUFFER
LON_MIN = float(city_lons.min()) - BBOX_BUFFER
LON_MAX = float(city_lons.max()) + BBOX_BUFFER

log.info(f'Loaded {len(latlong_df)} cities')
log.info(f'Bounding box: lat [{LAT_MIN:.2f}, {LAT_MAX:.2f}], lon [{LON_MIN:.2f}, {LON_MAX:.2f}]')
display(latlong_df.head())

In [ ]:
# ── Cell 5: GCS client ────────────────────────────────────────────────────────
storage_client = storage.Client.from_service_account_json(SA_KEY_FILE)
bucket = storage_client.bucket(BUCKET_NAME)
log.info('GCS client ready')

In [ ]:
# ── Cell 6: Checkpoint helpers ────────────────────────────────────────────────
def load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        return set(CHECKPOINT_FILE.read_text().splitlines())
    return set()

def save_checkpoint(blob_name: str):
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(blob_name + '\n')

PROCESSED_BLOBS = load_checkpoint()
log.info(f'Already processed: {len(PROCESSED_BLOBS)} blobs')

In [ ]:
# ── Cell 7: Core pipeline functions ───────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
def download_and_extract_grib(blob_obj) -> str:
    """
    Downloads a GCS blob into a unique per-blob subdirectory and extracts it.
    Returns the full path of the extracted .grib2 file.

    Key fixes vs original:
    - Uses download_as_bytes() to avoid Databricks PermissionError on os.utime()
    - Calls extractall() in a single forward pass (no getmembers() before it)
      to prevent EOFError on streaming gzip tarfiles
    - Each blob gets its own subdirectory so parallel workers never collide
    """
    tar_filename = blob_obj.name.split('/')[-1]
    work_dir = DOWNLOAD_DIR / tar_filename.replace('.tar.gz', '')
    work_dir.mkdir(parents=True, exist_ok=True)
    tar_path = work_dir / tar_filename

    log.info(f'[{tar_filename}] Downloading ...')
    t0 = time.time()

    # download_as_bytes() avoids the PermissionError that download_to_filename()
    # causes on Databricks (it tries os.utime() which is not permitted)
    raw = blob_obj.download_as_bytes()
    tar_path.write_bytes(raw)
    del raw   # free RAM immediately
    log.info(f'[{tar_filename}] Downloaded in {time.time()-t0:.1f}s')

    # Single forward pass through the gzip stream — never call getmembers() first
    # on a streaming tarfile as it exhausts the stream and extractall() then fails
    with tarfile.open(str(tar_path), 'r:gz') as tar:
        tar.extractall(path=str(work_dir))

    tar_path.unlink(missing_ok=True)   # free disk space

    # Delete any stale .idx files so cfgrib doesn't warn about them
    for idx in work_dir.glob('*.idx'):
        idx.unlink(missing_ok=True)

    grib_files = list(work_dir.glob('**/*.grib2'))
    if not grib_files:
        grib_files = [f for f in work_dir.iterdir() if f.is_file()]
    if not grib_files:
        raise FileNotFoundError(f'No GRIB file found after extracting {tar_filename}')

    grib_path = str(grib_files[0])
    log.info(f'[{tar_filename}] Extracted → {grib_path}')
    return grib_path


# ─────────────────────────────────────────────────────────────────────────────
def extract_all_cities_from_dataset(ds: xr.Dataset) -> pd.DataFrame:
    """
    Vectorized interpolation: interpolates ALL cities in a single xarray call
    per variable instead of looping 300 times.
    Shape of result: (time × step × city) flattened into rows.
    """
    frames = []
    for var in ds.data_vars:
        if var not in VARIABLES_MAPPING:
            continue
        col = VARIABLES_MAPPING[var]
        log.info(f'    Interpolating: {var} → {col}')

        # One call for all 300 cities — xarray vectorises this at C level
        interpolated = ds[var].interp(
            latitude=city_lats,
            longitude=city_lons,
            method='linear'
        )
        df = interpolated.to_dataframe(name=col).reset_index()
        keep = [c for c in ['time', 'valid_time', 'city', col] if c in df.columns]
        df = df[keep].rename(columns={'time': 'source_timestamp', 'valid_time': 'forecast_timestamp'})
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    result = frames[0]
    for df in frames[1:]:
        result = result.merge(df, on=['source_timestamp', 'forecast_timestamp', 'city'], how='outer')
    return result


# ─────────────────────────────────────────────────────────────────────────────
def process_grib_file(grib_path: str) -> pd.DataFrame:
    """
    Opens a GRIB file, slices it to the India bounding box, loads into RAM,
    and runs vectorized interpolation for all cities.

    Critical optimization: slice to India bbox BEFORE .load()
    The full global GRIB grid is ~4M points. India is ~57K points.
    Loading only the subset reduces RAM usage and load time by ~99%.
    (Previously: 1-2 hours to load. After fix: seconds.)
    """
    log.info(f'Opening GRIB: {grib_path}')
    t0 = time.time()

    # Only delete zero-byte (corrupt) .idx files — valid ones save ~13 min on reopen
    for idx in Path(grib_path).parent.glob('*.idx'):
        try:
            if idx.stat().st_size == 0:
                idx.unlink()
                log.info(f'  Removed corrupt (empty) index: {idx.name}')
        except Exception:
            pass

    datasets = cfgrib.open_datasets(grib_path)
    log.info(f'GRIB opened ({len(datasets)} datasets) in {time.time()-t0:.1f}s')

    all_frames = []

    for i, ds in enumerate(datasets):
        if i not in TARGET_DS_INDICES:
            continue

        log.info(f'  Dataset {i} | vars: {list(ds.data_vars)}')
        t1 = time.time()

        # Identify lat/lon dimension names (vary across GRIB producers)
        lat_dim = 'latitude'  if 'latitude'  in ds.dims else [d for d in ds.dims if 'lat' in d.lower()][0]
        lon_dim = 'longitude' if 'longitude' in ds.dims else [d for d in ds.dims if 'lon' in d.lower()][0]

        # Slice to India bounding box — GRIB latitudes can be descending, handle both
        lats = ds[lat_dim].values
        if lats[0] > lats[-1]:   # descending (most common in GRIB)
            ds_sub = ds.sel({lat_dim: slice(LAT_MAX, LAT_MIN), lon_dim: slice(LON_MIN, LON_MAX)})
        else:                      # ascending
            ds_sub = ds.sel({lat_dim: slice(LAT_MIN, LAT_MAX), lon_dim: slice(LON_MIN, LON_MAX)})

        total_pts  = ds[lat_dim].size  * ds[lon_dim].size
        subset_pts = ds_sub[lat_dim].size * ds_sub[lon_dim].size
        log.info(f'  Bbox subset: {subset_pts:,} pts of {total_pts:,} ({100*subset_pts/total_pts:.1f}%) — loading ...')

        ds = ds_sub.load()   # load only the small regional subset into RAM
        log.info(f'  Loaded in {time.time()-t1:.1f}s')

        df = extract_all_cities_from_dataset(ds)
        if not df.empty:
            all_frames.append(df)
        ds.close()

    if not all_frames:
        return pd.DataFrame()

    combined = all_frames[0]
    for df in all_frames[1:]:
        combined = combined.merge(
            df, on=['source_timestamp', 'forecast_timestamp', 'city'],
            how='outer', suffixes=('', '_dup')
        )
    combined = combined[[c for c in combined.columns if not c.endswith('_dup')]]
    return combined


# ─────────────────────────────────────────────────────────────────────────────
def process_blob(blob_obj) -> str:
    """
    Full pipeline for one GCS blob:
      download → extract → spatial subset → load → interpolate → push DB → cleanup
    Returns blob name on success, raises on failure.
    """
    blob_name = blob_obj.name
    t_start   = time.time()
    work_dir  = None

    try:
        # Special case: 20230611 file is already extracted
        if '20230611_00Z.tar.gz' in blob_name:
            grib_file = str(DOWNLOAD_DIR / 'fcst_20230611.grib2')
        else:
            grib_file = download_and_extract_grib(blob_obj)
            work_dir  = Path(grib_file).parent

        result_df = process_grib_file(grib_file)

        if result_df.empty:
            log.warning(f'No data extracted from {blob_name}')
            return blob_name

        # Attach City / State labels using the integer city index
        result_df = result_df.merge(city_meta, on='city', how='left').drop(columns='city')

        # Single batch push — not 300 individual calls
        log.info(f'Pushing {len(result_df):,} rows → european_weather_forecast')
        push_data(result_df, 'european_weather_forecast', 'append')

        # Cleanup extracted files
        if work_dir is not None and work_dir.exists():
            shutil.rmtree(str(work_dir), ignore_errors=True)

        elapsed = time.time() - t_start
        log.info(f'✅ Done: {blob_name} | {len(result_df):,} rows | {elapsed/60:.1f} min')
        return blob_name

    except Exception:
        log.error(f'❌ FAILED: {blob_name}\n{traceback.format_exc()}')
        raise


log.info('All pipeline functions defined ✓')

In [ ]:
# ── Cell 8: List blobs ────────────────────────────────────────────────────────
log.info('Listing GCS blobs ...')
all_blobs = [
    b for b in bucket.list_blobs(prefix=FOLDER_PATH)
    if (not YEAR_FILTER or YEAR_FILTER in b.name) and b.name.endswith('.tar.gz')
]

pending_blobs = [b for b in all_blobs if b.name not in PROCESSED_BLOBS]

log.info(f'Total blobs  : {len(all_blobs)}')
log.info(f'Done (skip)  : {len(all_blobs) - len(pending_blobs)}')
log.info(f'To process   : {len(pending_blobs)}')
log.info(f'Workers      : {MAX_WORKERS}')

est_min_per_blob = 25   # conservative estimate after bbox fix
est_hrs = (len(pending_blobs) * est_min_per_blob) / (MAX_WORKERS * 60)
log.info(f'Estimated time: {est_hrs:.1f} hrs (at ~{est_min_per_blob} min/blob × {MAX_WORKERS} workers)')

In [ ]:
# ── Cell 9: Run parallel extraction ──────────────────────────────────────────
success_count = 0
fail_count    = 0
failed_blobs  = []
t_total       = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {executor.submit(process_blob, b): b for b in pending_blobs}

    for future in as_completed(future_map):
        blob_obj = future_map[future]
        try:
            blob_name = future.result()
            save_checkpoint(blob_name)
            success_count += 1
        except Exception:
            fail_count += 1
            failed_blobs.append(blob_obj.name)

        done = success_count + fail_count
        elapsed_hr = (time.time() - t_total) / 3600
        remaining  = len(pending_blobs) - done
        rate       = done / elapsed_hr if elapsed_hr > 0 else 0
        eta_hr     = remaining / rate if rate > 0 else 0
        log.info(
            f'Progress: {done}/{len(pending_blobs)} '
            f'(✅ {success_count}  ❌ {fail_count}) | '
            f'elapsed {elapsed_hr:.1f}h | ETA {eta_hr:.1f}h'
        )

total_elapsed = time.time() - t_total
log.info('=' * 60)
log.info(f'FINISHED | Total: {total_elapsed/3600:.2f} hrs | ✅ {success_count} | ❌ {fail_count}')
if failed_blobs:
    log.warning('Failed blobs:\n' + '\n'.join(failed_blobs))

In [ ]:
# ── Cell 10: Retry failed blobs (run after Cell 9 if any failures) ───────────
if failed_blobs:
    log.info(f'Retrying {len(failed_blobs)} failed blobs sequentially ...')
    retry_map = {b.name: b for b in all_blobs}

    for blob_name in failed_blobs:
        blob_obj = retry_map.get(blob_name)
        if not blob_obj:
            log.warning(f'Blob not found in listing: {blob_name}')
            continue
        log.info(f'Retrying: {blob_name}')
        try:
            process_blob(blob_obj)
            save_checkpoint(blob_name)
            log.info(f'✅ Retry success: {blob_name}')
        except Exception as e:
            log.error(f'❌ Retry failed: {blob_name} — {e}')
else:
    log.info('No failed blobs to retry ✓')